# Multi-category query understanding with Jev and BM25

This notebook walks through the experiment in `configs/ecom_class/ecom_query_understanding_category_hierarchy_bm25_boosted_jev_choice_multiple.yml`. The CLI command that runs the full benchmark is:

```bash
uv run run --strategy configs/ecom_class/ecom_query_understanding_category_hierarchy_bm25_boosted_jev_choice_multiple.yml --dataset wands --workers 4
```

The strategy asks Jev to assign probabilities to popular full category paths, keeps every path with probability above `0.4`, runs BM25 over product titles and descriptions, and adds `10` to documents matching any selected path. This notebook rebuilds those steps directly; it does not import helpers from this repository.

## Install pinned dependencies

`cheat-at-search` supplies WANDS data, the generic search strategy interface, and ranking metrics. Jev is accessed through TypeSafe's SDK. The dependency versions are pinned so the data and interfaces remain reproducible.

In [1]:
!pip install -q git+https://github.com/softwaredoug/cheat-at-search.git@1ee82acd943135997b625846525e7f44329f55c8
!pip install -q typesafe-sdk==0.7.1


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Mount the shared search data directory

In Colab, mounting Google Drive keeps the dataset and API keys between sessions. If Drive is unavailable, the fallback uses a local cache.

In [2]:
from pathlib import Path

from cheat_at_search.data_dir import mount

try:
    mount(use_gdrive=True)
except ImportError:
    manual_path = str(Path.home() / '.search-experiments' / 'cheat-at-search')
    mount(use_gdrive=False, manual_path=manual_path)

2026-10-06 11:29:19,681 - cheat_at_search.data_dir - ERROR - Google Colab drive module not found. Ensure you're running this in Google Colab.


## Load the Jev key and WANDS data

`key_for_provider` reads `TYPESAFE_API_KEY` or prompts to save a key in the mounted data directory. WANDS includes the full `category hierarchy` field, product text, and query-to-product relevance judgments.

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd

from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus as wands_corpus
from cheat_at_search.wands_data import judgments as wands_judgments
from typesafe_sdk import Choice, TypeSafeClient

TYPESAFE_KEY = key_for_provider('typesafe')
corpus = wands_corpus.reset_index(drop=True).copy()
judgments = wands_judgments.reset_index(drop=True).copy()

corpus[['doc_id', 'title', 'category hierarchy']].head(3)

## Build Jev's option vocabulary

The query-understanding strategy first orders category paths by corpus frequency and caps that vocabulary at 300. Jev Choice supports at most 255 options, so this experiment passes the 255 most common paths. There is no synthetic `Unknown` option: unlike a single-choice classifier, thresholding can return no paths.

In [ ]:
CATEGORY_FIELD = 'category hierarchy'
CATEGORY_VOCABULARY_LIMIT = 300
JEV_CHOICE_LIMIT = 255
THRESHOLD = 0.4
PROMPT_TEMPLATE = (
    'Select all relevant values from the {field}. '
    'Category values use / separators; return full paths.\n\n{query}'
)

category_counts = (
    corpus[CATEGORY_FIELD]
    .dropna()
    .astype(str)
    .loc[lambda values: values != '']
    .value_counts()
)
category_vocabulary = category_counts.head(CATEGORY_VOCABULARY_LIMIT).index.tolist()
choice_values = category_vocabulary[:JEV_CHOICE_LIMIT]
CHOICE_CRITERIA = {category: None for category in choice_values}

pd.DataFrame({
    'category_path': choice_values[:10],
    'document_count': [int(category_counts[path]) for path in choice_values[:10]],
})

## Inspect one Jev classification

A Choice response contains a probability for every option. The enricher returns all category paths whose probability is **strictly greater than** `0.4`; a path exactly at the threshold is excluded.

In [ ]:
jev = TypeSafeClient(api_key=TYPESAFE_KEY, model='jev-latest')
classification_cache = {}

def classify_query(query, client=jev):
    if query not in classification_cache:
        response = client.system_one(
            state=query,
            questions={
                CATEGORY_FIELD: Choice(
                    instructions=PROMPT_TEMPLATE.format(
                        field=CATEGORY_FIELD, query=query
                    ),
                    criteria=CHOICE_CRITERIA,
                )
            },
        )
        answer = response.choices[CATEGORY_FIELD]
        probabilities = dict(answer.probabilities)
        categories = sorted(
            category
            for category, probability in probabilities.items()
            if category in CHOICE_CRITERIA
            and probability > THRESHOLD
        )
        classification_cache[query] = {
            'categories': categories,
            'probabilities': probabilities,
        }
    return classification_cache[query]

example_query = judgments['query'].drop_duplicates().iloc[0]
example_classification = classify_query(example_query)
example_classification['categories'], (
    pd.Series(example_classification['probabilities'], name='probability')
    .sort_values(ascending=False)
    .head(10)
)

## Index product text and category paths

WANDS already includes snowball indexes for titles and descriptions. We build the additional index for the complete hierarchy strings, because `bm25_boosted` checks whether each selected full category path matches a product's category field. This is a whole-path match boost, not a separate decaying boost at each taxonomy level.

In [ ]:
from searcharray import SearchArray
from searcharray.similarity import bm25_similarity

from cheat_at_search.strategy import SearchStrategy
from cheat_at_search.tokenizers import snowball_tokenizer

for field in ['title', 'description', CATEGORY_FIELD]:
    index_name = f'{field}_snowball'
    if index_name not in corpus.columns:
        corpus[index_name] = SearchArray.index(
            corpus[field].fillna('').astype(str), snowball_tokenizer
        )

corpus[[f'{CATEGORY_FIELD}_snowball']].head(2)

## Recreate the BM25 boosted strategy

The config combines title BM25 scores weighted by `9.4` and description scores weighted by `4`. Each selected category path contributes a fixed `+10` to matching documents. If multiple paths match the same document, their boosts add. The score otherwise remains ordinary BM25 relevance.

In [ ]:
class JevCategoryHierarchyBM25Boosted(SearchStrategy):
    def __init__(
        self, corpus, client, field_weights, boost_matches, workers=4
    ):
        super().__init__(corpus, top_k=10, workers=workers)
        self.index = corpus
        self.client = client
        self.field_weights = dict(field_weights)
        self.boost_matches = boost_matches
        self.category_index_name = f'{CATEGORY_FIELD}_snowball'
        self.similarity = bm25_similarity(k1=1.2, b=0.75)

    def classify(self, query):
        return classify_query(query, client=self.client)

    def _category_matches(self, category_path):
        terms = snowball_tokenizer(category_path)
        if not terms:
            return np.zeros(len(self.index), dtype=bool)
        return self.index[self.category_index_name].array.score(terms) > 0

    def search(self, query, k=10):
        scores = np.zeros(len(self.index), dtype=float)
        for term in snowball_tokenizer(query):
            for field, weight in self.field_weights.items():
                scores += (
                    self.index[f'{field}_snowball'].array.score(
                        term, similarity=self.similarity
                    )
                    * weight
                )

        for category_path in self.classify(query)['categories']:
            scores[self._category_matches(category_path)] += self.boost_matches

        top_indices = np.argsort(-scores)[:k]
        return top_indices, scores[top_indices]

    @property
    def cache_key(self):
        payload = {
            'type': 'jev_choice_multiple_bm25_boosted',
            'category_field': CATEGORY_FIELD,
            'choice_criteria': CHOICE_CRITERIA,
            'threshold': THRESHOLD,
            'prompt': PROMPT_TEMPLATE,
            'model': 'jev-latest',
            'field_weights': self.field_weights,
            'boost_matches': self.boost_matches,
        }
        encoded = json.dumps(payload, sort_keys=True).encode('utf-8')
        return hashlib.md5(encoded).hexdigest()

strategy = JevCategoryHierarchyBM25Boosted(
    corpus=corpus,
    client=jev,
    field_weights={'title': 9.4, 'description': 4.0},
    boost_matches=10,
    workers=4,
)

strategy.classify(example_query)['categories']

## Inspect ranked results for one query

This query reuses the classification above. The displayed score includes the category boost as well as the baseline text relevance.

In [ ]:
top_indices, top_scores = strategy.search(example_query, k=5)
preview = corpus.iloc[top_indices][['doc_id', 'title', CATEGORY_FIELD]].copy()
preview['score'] = top_scores
preview

## Benchmark against baseline BM25

`run_strategy` runs the same ranked-search interface used by the CLI and grades results against WANDS judgments. Start with a small query sample to limit Jev calls; set `QUERY_LIMIT = None` to process the full judgment set like the command at the top. Both strategies use the same seed and query sample.

In [ ]:
from cheat_at_search.search import mrrs, ndcgs, run_strategy
from cheat_at_search.strategy import BM25Search

QUERY_LIMIT = 8  # Set to None for all WANDS judgment queries.
QUERY_SEED = 42

baseline = BM25Search(
    corpus, title_boost=9.4, description_boost=4.0
)
baseline.workers = 4
baseline_graded = run_strategy(
    baseline, judgments, num_queries=QUERY_LIMIT, seed=QUERY_SEED, cache=True
)
boosted_graded = run_strategy(
    strategy, judgments, num_queries=QUERY_LIMIT, seed=QUERY_SEED, cache=True
)

metrics = pd.DataFrame(
    {
        'BM25 baseline': [ndcgs(baseline_graded).mean(), mrrs(baseline_graded).mean()],
        'Jev category boosted': [ndcgs(boosted_graded).mean(), mrrs(boosted_graded).mean()],
    },
    index=['Mean NDCG@10', 'Mean MRR'],
)
metrics

## What this experiment tests

The classifier can return several category paths when their Jev probabilities each exceed `0.4`. BM25 still determines the text-relevance ordering; category matches add a fixed boost and may lift matching products. The benchmark compares that combined strategy with unboosted BM25 using NDCG@10 and MRR. The full CLI command uses all judgments and four workers; the notebook sample is deliberately smaller for an interactive first run.